## Kernelemente (Core Entities) 

Für unser neues Projekt "Lemon Shop" wollen wir folgendes System erstellen

    lemon-shop (System)
    │
    ├── shop-frontend
    │   └── shop-api
    │
    ├── shop-api
    │   ├── product-service
    │   ├── order-service
    │   └── payment-service
    │
    ├── product-service
    │
    ├── order-service
    │   ├── product-service
    │   └── payment-service
    │
    └── payment-service
        
---

Um die Konfiguration von Backstage nicht ändern zu müssen  überschreiben wir die bestehende `~/lemon-shop/examples/entities.yaml` Datei.

Die Aufteilung in mehrere Dateien hält die einzelnen Bestandteile übersichtlich, erleichtert gezielte Änderungen und verhindert, dass bei jeder Anpassung die gesamte Katalogdefinition bearbeitet werden muss. 

Die zentrale `entities.yaml` dient dabei nur als Einstiegspunkt und verweist auf die separat gepflegten System-, Component-, API- und Resource-Dateien.


In [ ]:
%%bash
cat > ~/lemon-shop/examples/entities.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: Location
metadata:
  name: lemon-shop-entities
  namespace: default
  title: Lemon Shop Entities
  description: Einstiegspunkt für alle Lemon-Shop Catalog Entities
spec:
  type: file
  targets:
    - ./lemon-shop.yaml
    - ./shop-frontend.yaml
    - ./shop-api.yaml
    - ./shop-api-definition.yaml
    - ./product-service.yaml
    - ./order-service.yaml
    - ./payment-service.yaml
    - ./database.yaml
    - ./kubernetes.yaml
EOF

### System

Ein System fasst alle zusammengehörenden Teile des Lemon Shops zu einer gemeinsamen Anwendung zusammen.


In [ ]:
%%bash
cat > ~/lemon-shop/examples/lemon-shop.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: System
metadata:
  name: lemon-shop
  namespace: default
  title: Lemon Shop
  description: Übergeordnetes System der Lemon-Shop-Applikation
  tags:
    - lemon-shop
    - ecommerce
spec:
  owner: shop-team
  lifecycle: experimental
EOF

### Components

Components sind die einzelnen Bausteine des Lemon Shops, die jeweils eine klar abgegrenzte Aufgabe übernehmen und gemeinsam das Gesamtsystem bilden.


In [ ]:
%%bash
cat > ~/lemon-shop/examples/shop-frontend.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: shop-frontend
  namespace: default
  title: Shop Frontend
  description: Webbasierte Benutzeroberfläche des Lemon Shops
  tags:
    - lemon-shop
    - frontend
    - web
spec:
  type: website
  lifecycle: experimental
  owner: shop-team
  system: lemon-shop
  dependsOn:
    - component:default/shop-api
    - resource:default/kubernetes
  consumesApis:
    - api:default/shop-api
EOF

cat > ~/lemon-shop/examples/shop-api.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: shop-api
  namespace: default
  title: Shop API
  description: Zentraler API-Einstiegspunkt für das Shop Frontend
  tags:
    - lemon-shop
    - python
    - fastapi
    - api
    - backend
spec:
  type: service
  lifecycle: experimental
  owner: shop-team
  system: lemon-shop
  providesApis:
    - api:default/shop-api
  dependsOn:
    - component:default/product-service
    - component:default/order-service
    - component:default/payment-service
EOF

cat > ~/lemon-shop/examples/product-service.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: product-service
  namespace: default
  title: Product Service
  description: Verwaltet Produkte, Preise und Produktinformationen
  tags:
    - lemon-shop
    - python
    - products
spec:
  type: service
  lifecycle: experimental
  owner: shop-team
  system: lemon-shop
  dependsOn:
    - resource:default/database
    - resource:default/kubernetes
EOF

cat > ~/lemon-shop/examples/order-service.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: order-service
  namespace: default
  title: Order Service
  description: Verwaltet Warenkörbe und Bestellungen
  tags:
    - lemon-shop
    - python
    - orders
spec:
  type: service
  lifecycle: experimental
  owner: shop-team
  system: lemon-shop
  dependsOn:
    - component:default/product-service
    - component:default/payment-service
    - resource:default/database
    - resource:default/kubernetes
EOF

cat > ~/lemon-shop/examples/payment-service.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: payment-service
  namespace: default
  title: Payment Service
  description: Verarbeitet Zahlungsvorgänge für Bestellungen
  tags:
    - lemon-shop
    - python
    - payments
spec:
  type: service
  lifecycle: experimental
  owner: backoffice-team
  system: lemon-shop
  dependsOn:
    - resource:default/database
    - resource:default/kubernetes
EOF

- - -
### Abhängigkeiten im Backstage-Katalog prüfen

Öffne in der Backstage-UI den Software Catalog und klicke auf das System **lemon-shop**. 

Wähle anschliessend die Component **order-service** aus und prüfe im Abhängigkeitsdiagramm die Beziehungen zu den übrigen Services.


In [ ]:
! echo "http://$(cat ~/data/server-ip):3000"